In [ ]:
from google.colab import files
files.upload()

Saving prompts.csv to prompts.csv


{'prompts.csv': b'id,version,word,english_meaning,prompt\r\n1,punjabi,\xd9\xbe\xd9\x84 \xda\xa9\xdb\x92 ,Tomorrow ,"What does the Pakistani Punjabi word ""\xd9\xbe\xd9\x84 \xda\xa9\xdb\x92 "" mean in English? Answer in one line."\r\n1,urdu,\xda\xa9\xd9\x84/\xd8\xa2\xd8\xac \xda\xa9\xd8\xa7 \xd8\xaf\xd9\x86 \xda\x86\xda\xbe\xd9\x88\xda\x91 \xda\xa9\xd8\xb1 ,Tomorrow ,"What does the Urdu word ""\xda\xa9\xd9\x84/\xd8\xa2\xd8\xac \xda\xa9\xd8\xa7 \xd8\xaf\xd9\x86 \xda\x86\xda\xbe\xd9\x88\xda\x91 \xda\xa9\xd8\xb1 "" mean in English? Answer in one line."\r\n2,punjabi,"\n\xd8\xac\xd9\x86\xd8\xaf\xd8\xb1\xd8\xa7",Lock ,"What does the Pakistani Punjabi word ""\n\xd8\xac\xd9\x86\xd8\xaf\xd8\xb1\xd8\xa7"" mean in English? Answer in one line."\r\n2,urdu,\xd8\xaa\xd8\xa7\xd9\x84\xd8\xa7 ,Lock ,"What does the Urdu word ""\xd8\xaa\xd8\xa7\xd9\x84\xd8\xa7 "" mean in English? Answer in one line."\r\n3,punjabi,"\n\xd8\xaf\xd9\x8f\xdb\x81\xd8\xa7\xd8\xac\xd9\x88",Remarried Person ,"What does the Pakistan

In [ ]:
import pandas as pd
df = pd.read_csv("prompts.csv").dropna(subset=["prompt"])
print(len(df), "sawal")
df.head()

80 sawal


,id,version,word,english_meaning,prompt
0,1,punjabi,پل کے,Tomorrow,"What does the Pakistani Punjabi word ""پل کے "" ..."
1,1,urdu,کل/آج کا دن چھوڑ کر,Tomorrow,"What does the Urdu word ""کل/آج کا دن چھوڑ کر ""..."
2,2,punjabi,\nجندرا,Lock,"What does the Pakistani Punjabi word ""\nجندرا""..."
3,2,urdu,تالا,Lock,"What does the Urdu word ""تالا "" mean in Englis..."
4,3,punjabi,\nدُہاجو,Remarried Person,"What does the Pakistani Punjabi word ""\nدُہاجو..."


In [ ]:
!pip -q install -U transformers accelerate

import pandas as pd, torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-E2B-it"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

# words se extra line-breaks saaf karein aur sawal dobara banayen
df = pd.read_csv("prompts.csv").dropna(subset=["prompt"])
df["word"] = df["word"].astype(str).str.replace(r"\\n|\n|\r", "", regex=True).str.strip()
lang = {"punjabi": "Pakistani Punjabi", "urdu": "Urdu"}
df["prompt"] = [f'What does the {lang[v]} word "{w}" mean in English? Answer in one line.'
                for v, w in zip(df["version"], df["word"])]

def ask(q):
    msgs = [{"role": "user", "content": [{"type": "text", "text": q}]}]
    inputs = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

answers = []
for i, q in enumerate(df["prompt"], 1):
    answers.append(ask(q))
    print(i, "/", len(df), "->", answers[-1][:80])

df["model_answer"] = answers
df["score"] = ""   # 1 / 0.5 / 0 — yeh aap khud bharenge
df["note"] = ""
df.to_csv("results.csv", index=False, encoding="utf-8-sig")

from google.colab import files
files.download("results.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.5 MB/s eta 0:00:00


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

1 / 80 -> "پل کے" in Pakistani Punjabi generally means "of the bridge" or "for the bridge.
2 / 80 -> "کل/آج کا دن چھوڑ کر" means "leaving yesterday/today's day."
3 / 80 -> The Pakistani Punjabi word "جندرا" (jandra) generally refers to a **place name**
4 / 80 -> The Urdu word "تالا" means "lock" in English.
5 / 80 -> The Pakistani Punjabi word "دُہاجو" (duhajo) generally means "to be lost" or "to
6 / 80 -> The Urdu phrase "دوسری شادی کرنے والا آدمی" means "a man who remarries."
7 / 80 -> The Pakistani Punjabi word "جواک" (jawaak) generally means **a small, insignific
8 / 80 -> The Urdu word "بچہ" means "child" in English.
9 / 80 -> "جھانجھریں" (jhanjharain) generally means "to be restless" or "to be agitated" i
10 / 80 -> The Urdu word "کھنک" (khank) generally means a **hiss** or a **squeak**.
11 / 80 -> "Bhagwan" in Punjabi means "God" or "Lord" in English.
12 / 80 -> The Urdu word "بیوی" means "wife" in English.
13 / 80 -> The Pakistani Punjabi word "کھرلی" (kharli) generally means *

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>